# SageMaker Local Training: Churn Model

This notebook runs the `train` container locally using SageMaker Local Mode.

Expected project layout:

```text
churn-prediction/
├── pyproject.toml
├── uv.lock
└── ml/
    ├── containers/
    │   ├── train.Dockerfile
    │   └── requirements-train.txt
    ├── outputs/
    │   ├── processed/
    │   │   ├── train.csv
    │   │   ├── valid.csv
    │   │   └── feature_columns.json
    │   ├── models/
    │   ├── reports/
    │   └── figures/
    │       └── model/
    └── src/
        └── churn_ml/
            └── train.py

## 1. Setup

Run this notebook from the repository root or set `PROJECT_ROOT` manually.

In [4]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# If your notebook is opened from ml/notebooks, uncomment this:
PROJECT_ROOT = Path.cwd().parents[1]

ML_DIR = PROJECT_ROOT / "ml"
OUTPUTS_DIR = ML_DIR / "outputs"
PROCESSED_DIR = OUTPUTS_DIR / "processed"

MODEL_DIR = OUTPUTS_DIR / "models"
REPORT_DIR = OUTPUTS_DIR / "reports"
FIGURE_DIR = OUTPUTS_DIR / "figures" / "model"

TRAIN_LOCAL = PROCESSED_DIR / "train.csv"
VALID_LOCAL = PROCESSED_DIR / "valid.csv"
FEATURE_COLUMNS_LOCAL = PROCESSED_DIR / "feature_columns.json"

print("PROJECT_ROOT:", PROJECT_ROOT)

print("TRAIN_LOCAL exists:", TRAIN_LOCAL.exists(), TRAIN_LOCAL)
print("VALID_LOCAL exists:", VALID_LOCAL.exists(), VALID_LOCAL)
print(
    "FEATURE_COLUMNS_LOCAL exists:",
    FEATURE_COLUMNS_LOCAL.exists(),
    FEATURE_COLUMNS_LOCAL,
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT: /Users/alex/churn-prediction
TRAIN_LOCAL exists: True /Users/alex/churn-prediction/ml/outputs/processed/train.csv
VALID_LOCAL exists: True /Users/alex/churn-prediction/ml/outputs/processed/valid.csv
FEATURE_COLUMNS_LOCAL exists: True /Users/alex/churn-prediction/ml/outputs/processed/feature_columns.json


## 2. Build the local Docker image

In [5]:
!docker build -f ../containers/train.Dockerfile -t churn-train:local ../..


[+] Building 0.0s (0/1)                                         docker:orbstack
[+] Building 0.2s (1/2)                                         docker:orbstack
 => [internal] load build definition from train.Dockerfile                 0.0s
 => => transferring dockerfile: 415B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.13-slim        0.2s
[+] Building 0.3s (1/2)                                         docker:orbstack
 => [internal] load build definition from train.Dockerfile                 0.0s
 => => transferring dockerfile: 415B                                       0.0s
 => [internal] load metadata for docker.io/library/python:3.13-slim        0.3s
[+] Building 0.3s (2/2)                                         docker:orbstack
 => [internal] load build definition from train.Dockerfile                 0.0s
 => => transferring dockerfile: 415B                                       0.0s
 => [internal] load metadata for docker

## 3. Smoke test with plain Docker

This is faster to debug than SageMaker Local Mode. If this fails, fix Docker/script paths before using SageMaker.

In [6]:
!docker run --rm \
  -v "{OUTPUTS_DIR}:/opt/program/outputs" \
  churn-train:local

Fitting 3 folds for each of 48 candidates, totalling 144 fits
{
  "roc_auc": 0.7273564508127708,
  "accuracy": 0.6673303939695634,
  "precision": 0.661750936329588,
  "recall": 0.7597420048374093,
  "confusion_matrix": [
    [
      1865,
      1445
    ],
    [
      894,
      2827
    ]
  ],
  "classification_report": {
    "0": {
      "precision": 0.6759695541862993,
      "recall": 0.5634441087613293,
      "f1-score": 0.6145987806887461,
      "support": 3310.0
    },
    "1": {
      "precision": 0.661750936329588,
      "recall": 0.7597420048374093,
      "f1-score": 0.7073689478293507,
      "support": 3721.0
    },
    "accuracy": 0.6673303939695634,
    "macro avg": {
      "precision": 0.6688602452579437,
      "recall": 0.6615930567993693,
      "f1-score": 0.6609838642590484,
      "support": 7031.0
    },
    "weighted avg": {
      "precision": 0.6684446676772932,
      "recall": 0.6673303939695634,
      "f1-score": 0.6636953234181145,
      "support": 7031.0
    }
  

## 4. Run with SageMaker Local Mode

This uses your already-built local image: `churn-train:local`.

SageMaker Training maps local inputs to `/opt/ml/input/data/<channel>` inside the container.

Training artifacts are written to:

- `/opt/ml/model`
- `/opt/ml/output/data`

Input channels used in this notebook:

- `train`
- `validation`
- `config`

In [12]:
import sagemaker
from sagemaker.estimator import Estimator
from sagemaker.local import LocalSession

local_session = LocalSession()
local_session.config = {"local": {"local_code": True}}

# Local mode does not need a real AWS execution role for the local Docker run.
role = "arn:aws:iam::000000000000:role/SageMakerLocalRole"

estimator = Estimator(
    image_uri="churn-train:local",
    role=role,
    instance_count=1,
    instance_type="local",
    sagemaker_session=local_session,
    entry_point="train.py",
    hyperparameters={
        "train-path": "/opt/ml/input/data/training/train.csv",
        "valid-path": "/opt/ml/input/data/training/valid.csv",
        "feature-columns-path": "/opt/ml/input/data/training/feature_columns.json",
        "model-dir": "/opt/ml/model",
        "report-dir": "/opt/ml/output/data/reports",
        "figure-dir": "/opt/ml/output/data/figures",
    },
)

estimator.fit(inputs=f"file://{Path(TRAIN_LOCAL).parent.resolve()}")

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: churn-train-2026-05-11-01-46-16-327
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-

time="2026-05-10T21:46:16-04:00" level=warning msg="/private/var/folders/2f/j192nttn58x0gl_t_c5x2j340000gn/T/tmpblwlnl7w/docker-compose.yaml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
time="2026-05-10T21:46:16-04:00" level=warning msg="a network with name sagemaker-local exists but was not created for project \"tmpblwlnl7w\".\nSet `external: true` to use an existing network"
 Container sl1l8rmk09-algo-1-reanl  Creating
 Container sl1l8rmk09-algo-1-reanl  Created
Attaching to sl1l8rmk09-algo-1-reanl
sl1l8rmk09-algo-1-reanl  | usage: train.py [-h] [--train-path TRAIN_PATH] [--valid-path VALID_PATH]
sl1l8rmk09-algo-1-reanl  |                 [--feature-columns-path FEATURE_COLUMNS_PATH]
sl1l8rmk09-algo-1-reanl  |                 [--model-dir MODEL_DIR] [--report-dir REPORT_DIR]
sl1l8rmk09-algo-1-reanl  |                 [--figure-dir FIGURE_DIR]
sl1l8rmk09-algo-1-reanl  | train.py: error: unrecognized arguments: train
sl1l8rmk0

ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:28                                                                                   │
│                                                                                                  │
│   25 │   },                                                                                      │
│   26 )                                                                                           │
│   27                                                                                             │
│ ❱ 28 estimator.fit(                                                                              │
│   29 │   inputs=f"file://{Path(TRAIN_LOCAL).parent.resolve()}"                                   │
│   30 )                                                                                           │
│   31                                                                                             │
│                                                                                                  │
│ /Users/alex/churn-prediction/.venv-sm/lib/python3.13/site-packages/sagemaker/telemetry/telemetry_loggi │
│ ng.py:166 in wrapper                                                                             │
│                                                                                                  │
│   163 │   │   │   │   │   caught_ex = e                                                          │
│   164 │   │   │   │   finally:                                                                   │
│   165 │   │   │   │   │   if caught_ex:                                                          │
│ ❱ 166 │   │   │   │   │   │   raise caught_ex                                                    │
│   167 │   │   │   │   │   return response  # pylint: disable=W0150                               │
│   168 │   │   │   else:                                                                          │
│   169 │   │   │   │   logger.debug(                                                              │
│                                                                                                  │
│ /Users/alex/churn-prediction/.venv-sm/lib/python3.13/site-packages/sagemaker/telemetry/telemetry_loggi │
│ ng.py:137 in wrapper                                                                             │
│                                                                                                  │
│   134 │   │   │   │   start_timer = perf_counter()                                               │
│   135 │   │   │   │   try:                                                                       │
│   136 │   │   │   │   │   # Call the original function                                           │
│ ❱ 137 │   │   │   │   │   response = func(*args, **kwargs)                                       │
│   138 │   │   │   │   │   stop_timer = perf_counter()                                            │
│   139 │   │   │   │   │   elapsed = stop_timer - start_timer                                     │
│   140 │   │   │   │   │   extra += f"&x-latency={round(elapsed, 2)}"                             │
│                                                                                                  │
│ /Users/alex/churn-prediction/.venv-sm/lib/python3.13/site-packages/sagemaker/workflow/pipeline_context │
│ .py:346 in wrapper                                                                               │
│                                                                                                  │
│   343 │   │   │                                                                                  │
│   344 │   │   │   return _StepArguments(retrieve_caller_name(self_instance), run_func, *args,    │
│   345 │   │                                                                                      │
│ ❱ 346 │   │   return run_func(*args, **kw

In [6]:
from importlib.metadata import version

print(sagemaker.__file__)
print(version("sagemaker"))

/Users/alex/churn-prediction/.venv-sm/lib/python3.13/site-packages/sagemaker/__init__.py
2.237.3


## 5. Inspect outputs

In [7]:
import json

for path in sorted(PROCESSED_DIR.glob("*")):
    print(path.name, path.stat().st_size, "bytes")

metadata_path = PROCESSED_DIR / "preprocess_metadata.json"
if metadata_path.exists():
    print("\nmetadata:")
    print(json.dumps(json.loads(metadata_path.read_text()), indent=2))

current.csv 5688372 bytes
feature_columns.json 422 bytes
preprocess_metadata.json 600 bytes
train.csv 2755851 bytes
valid.csv 688779 bytes

metadata:
{
  "n_train_rows": 28120,
  "n_valid_rows": 7031,
  "n_current_rows": 59818,
  "n_features": 12,
  "target_column": "is_churn",
  "feature_columns": [
    "n_ebookdownloaded_last_28d",
    "n_readingownedbook_last_28d",
    "n_readingfreepreview_last_28d",
    "n_searchmade_last_28d",
    "n_highlightcreated_last_28d",
    "n_bookmarkcreated_last_28d",
    "n_total_events_last_90d",
    "n_unique_products_last_90d",
    "n_free_content_actions_last_28d",
    "n_enhanced_reading_actions_last_28d",
    "pct_reading_owned_book_events_last_28d",
    "n_downloads_per_unique_product_last_90d"
  ]
}
